# 00 Data Inventory

Data Inventory — list and describe all raw files

## Setup

In [1]:
import sys
from pathlib import Path

# Find FinSense project root automatically
current = Path.cwd().resolve()

for path in [current, *current.parents]:
    if (path / "src").is_dir():
        ROOT = path
        break
else:
    raise RuntimeError("Could not find FinSense project root containing 'src/'")

# Add project root to Python path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Python:", sys.executable)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import load_config
from src.utils.paths import ROOT, DATA_RAW, DATA_PROCESSED
from src.utils.seed import set_seed

set_seed(42)

%matplotlib inline

Project root: /Users/ms/Desktop/FinSense
Python: /Users/ms/Desktop/FinSense/.venv/bin/python


In [2]:
import sys
print(sys.executable)

/Users/ms/Desktop/FinSense/.venv/bin/python


In [3]:
from pathlib import Path
import pandas as pd
import os
import sys

ROOT = Path.cwd().parent

RAW = ROOT / "data" / "raw"

IBM_DIR = RAW / "IBM"
FINANCIAL_DIR = RAW / "financial"
PAYSIM_DIR = RAW / "paysim"

print("Project root:", ROOT)
print("IBM:", IBM_DIR)
print("Financial:", FINANCIAL_DIR)
print("PaySim:", PAYSIM_DIR)

Project root: /Users/ms/Desktop/FinSense
IBM: /Users/ms/Desktop/FinSense/data/raw/IBM
Financial: /Users/ms/Desktop/FinSense/data/raw/financial
PaySim: /Users/ms/Desktop/FinSense/data/raw/paysim


## 2. Dataset Registry

Before loading any data, we define the location of every raw dataset.

Raw data is treated as immutable source data.
All cleaning and transformation will happen downstream.

In [4]:
datasets = {
    "IBM_transactions": IBM_DIR / "credit_card_transactions-ibm_v2.csv",
    "IBM_users": IBM_DIR / "sd254_users.csv",
    "IBM_cards": IBM_DIR / "sd254_cards.csv",
    "IBM_user0_transactions": IBM_DIR / "User0_credit_card_transactions.csv",
    "Financial": FINANCIAL_DIR / "cleaned_financial_data.csv",
    "PaySim": PAYSIM_DIR / "PaySim.csv",
}
for name, path in datasets.items():
    print(f"{name:30} | exists={path.exists()} | size={path.stat().st_size / (1024**2):.2f} MB")

IBM_transactions               | exists=True | size=2241.84 MB
IBM_users                      | exists=True | size=0.21 MB
IBM_cards                      | exists=True | size=0.46 MB
IBM_user0_transactions         | exists=True | size=1.81 MB
Financial                      | exists=True | size=1.28 MB
PaySim                         | exists=True | size=470.67 MB


In [5]:
import io

def load_fast(csv_path, sample_rows=None):
    """Load a dataset preferring Parquet if available, else CSV sample."""
    proc = ROOT / 'data' / 'processed'
    parquet_path = proc / (csv_path.stem.replace('cleaned_financial_data','financial')
                                       .replace('credit_card_transactions-ibm_v2','ibm_transactions')
                                       .replace('PaySim','paysim') + '.parquet')

    if parquet_path.exists():
        print(f'  Loading from Parquet: {parquet_path.name}')
        df = pd.read_parquet(parquet_path)
        if sample_rows:
            df = df.head(sample_rows)
        return df

    # Fallback: fast line-by-line CSV read
    rows = sample_rows or 5_000
    print(f'  No parquet found — reading first {rows:,} rows from CSV (run scripts/convert_to_parquet.py for full speed)')
    with open(csv_path, 'rb') as f:
        lines = [f.readline() for _ in range(rows + 1)]
    return pd.read_csv(io.BytesIO(b''.join(lines)), low_memory=False)


def profile_csv(path, sample_rows=5_000):
    """Profile a dataset — instant via Parquet, fast-sampled via CSV."""
    print(f"\n{'=' * 70}")
    print(f"DATASET: {path.name}")
    print(f"{'=' * 70}")
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"CSV size: {size_mb:.2f} MB")

    df = load_fast(path, sample_rows=sample_rows if size_mb > 100 else None)

    print(f"\nLoaded shape: {df.shape}")
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nDtypes:")
    print(df.dtypes.to_string())
    print("\nMissing values:")
    missing = df.isna().sum()
    print(missing[missing > 0].to_string() if missing.any() else "  None")
    print("\nSample (first 5 rows):")
    display(df.head())
    return df


In [6]:
financial_file = FINANCIAL_DIR / "cleaned_financial_data.csv"
paysim_file = PAYSIM_DIR / "PaySim.csv"
ibm_file = IBM_DIR / "credit_card_transactions-ibm_v2.csv"

In [7]:
financial = load_fast(financial_file)
print('Full financial shape:', financial.shape)
profile_csv(financial_file)


  Loading from Parquet: financial.parquet
Full financial shape: (8970, 22)

DATASET: cleaned_financial_data.csv
CSV size: 1.28 MB
  Loading from Parquet: financial.parquet

Loaded shape: (8970, 22)

Columns:
['user_id', 'Category', 'Amount', 'age', 'gender', 'education_level', 'employment_status', 'job_title', 'monthly_income_usd', 'monthly_expenses_usd', 'savings_usd', 'has_loan', 'loan_type', 'loan_amount_usd', 'loan_term_months', 'monthly_emi_usd', 'loan_interest_rate_pct', 'debt_to_income_ratio', 'credit_score', 'savings_to_income_ratio', 'region', 'record_date']

Dtypes:
user_id                        str
Category                       str
Amount                     float64
age                          int64
gender                         str
education_level                str
employment_status              str
job_title                      str
monthly_income_usd         float64
monthly_expenses_usd       float64
savings_usd                float64
has_loan                       s

,user_id,Category,Amount,age,gender,education_level,employment_status,job_title,monthly_income_usd,monthly_expenses_usd,...,loan_type,loan_amount_usd,loan_term_months,monthly_emi_usd,loan_interest_rate_pct,debt_to_income_ratio,credit_score,savings_to_income_ratio,region,record_date
0,U32234,Health & Fitness,282.25,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
1,U32234,Food & Drink,1082.58,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
2,U32234,Entertainment,1031.44,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
3,U32234,Shopping,1782.34,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
4,U28077,Entertainment,463.71,43,Female,Bachelor,Employed,Driver,3231.62,1715.77,...,Business,493968.77,24,22787.31,9.97,7.05,698,4.91,North America,2021-12-23


,user_id,Category,Amount,age,gender,education_level,employment_status,job_title,monthly_income_usd,monthly_expenses_usd,...,loan_type,loan_amount_usd,loan_term_months,monthly_emi_usd,loan_interest_rate_pct,debt_to_income_ratio,credit_score,savings_to_income_ratio,region,record_date
0,U32234,Health & Fitness,282.25,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
1,U32234,Food & Drink,1082.58,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
2,U32234,Entertainment,1031.44,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
3,U32234,Shopping,1782.34,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
4,U28077,Entertainment,463.71,43,Female,Bachelor,Employed,Driver,3231.62,1715.77,...,Business,493968.77,24,22787.31,9.97,7.05,698,4.91,North America,2021-12-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8965,U17955,Shopping,481.83,56,Female,Bachelor,Employed,Accountant,9930.58,8833.99,...,No Loan,0.00,0,0.00,0.00,0.00,648,7.84,Asia,2024-08-23
8966,U17955,Travel,732.02,56,Female,Bachelor,Employed,Accountant,9930.58,8833.99,...,No Loan,0.00,0,0.00,0.00,0.00,648,7.84,Asia,2024-08-23
8967,U17955,Food & Drink,2636.60,56,Female,Bachelor,Employed,Accountant,9930.58,8833.99,...,No Loan,0.00,0,0.00,0.00,0.00,648,7.84,Asia,2024-08-23
8968,U17955,Rent,183.25,56,Female,Bachelor,Employed,Accountant,9930.58,8833.99,...,No Loan,0.00,0,0.00,0.00,0.00,648,7.84,Asia,2024-08-23


In [8]:
paysim_sample = profile_csv(paysim_file)



DATASET: PaySim.csv
CSV size: 470.67 MB
  Loading from Parquet: paysim.parquet

Loaded shape: (5000, 11)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Dtypes:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64

Missing values:
  None

Sample (first 5 rows):


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [9]:
# IBM: instant if parquet exists, else reads 5k row sample
ibm_sample = profile_csv(ibm_file)



DATASET: credit_card_transactions-ibm_v2.csv
CSV size: 2241.84 MB
  Loading from Parquet: ibm_transactions.parquet

Loaded shape: (5000, 15)

Columns:
['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']

Dtypes:
User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time                  str
Amount                str
Use Chip              str
Merchant Name       int64
Merchant City         str
Merchant State        str
Zip               float64
MCC                 int64
Errors?               str
Is Fraud?             str

Missing values:
Merchant State     295
Zip                367
Errors?           4874

Sample (first 5 rows):


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


In [10]:
dataset_summary = pd.DataFrame([
    {
        "dataset": "Financial",
        "level": "User",
        "primary_role": "Financial profile",
        "scale": "Small",
        "granularity": "User-level"
    },
    {
        "dataset": "IBM",
        "level": "Transaction",
        "primary_role": "Spending behavior",
        "scale": "Very large",
        "granularity": "Transaction-level"
    },
    {
        "dataset": "PaySim",
        "level": "Transaction",
        "primary_role": "Transaction/cash-flow behavior",
        "scale": "Large",
        "granularity": "Transaction-level"
    }
])

display(dataset_summary)

,dataset,level,primary_role,scale,granularity
0,Financial,User,Financial profile,Small,User-level
1,IBM,Transaction,Spending behavior,Very large,Transaction-level
2,PaySim,Transaction,Transaction/cash-flow behavior,Large,Transaction-level


In [11]:
print("FINANCIAL:", financial.shape)
print("PAYSIM SAMPLE:", paysim_sample.shape)
print("IBM SAMPLE:", ibm_sample.shape)

print("\nFINANCIAL COLUMNS:")
print(financial.columns.tolist())

print("\nPAYSIM COLUMNS:")
print(paysim_sample.columns.tolist())

print("\nIBM COLUMNS:")
print(ibm_sample.columns.tolist())

FINANCIAL: (8970, 22)
PAYSIM SAMPLE: (5000, 11)
IBM SAMPLE: (5000, 15)

FINANCIAL COLUMNS:
['user_id', 'Category', 'Amount', 'age', 'gender', 'education_level', 'employment_status', 'job_title', 'monthly_income_usd', 'monthly_expenses_usd', 'savings_usd', 'has_loan', 'loan_type', 'loan_amount_usd', 'loan_term_months', 'monthly_emi_usd', 'loan_interest_rate_pct', 'debt_to_income_ratio', 'credit_score', 'savings_to_income_ratio', 'region', 'record_date']

PAYSIM COLUMNS:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

IBM COLUMNS:
['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']


In [12]:
financial.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,8970,2746,U15106,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,8970,7,Health & Fitness,1417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Amount,8970.0,NaN,NaN,NaN,980.878169,847.782968,18.18,402.1075,724.58,1289.9025,7461.66
age,8970.0,NaN,NaN,NaN,43.288183,15.056115,18.0,31.0,43.0,56.0,69.0
gender,8970,3,Female,4329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education_level,8970,5,Bachelor,3722,NaN,NaN,NaN,NaN,NaN,NaN,NaN
employment_status,8970,4,Employed,5373,NaN,NaN,NaN,NaN,NaN,NaN,NaN
job_title,8970,9,Accountant,1094,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_income_usd,8970.0,NaN,NaN,NaN,4825.722931,1823.350735,500.0,3547.43,4766.05,6070.69,10222.85
monthly_expenses_usd,8970.0,NaN,NaN,NaN,3073.900326,1433.385067,150.41,1987.76,2958.35,3956.77,8833.99


In [13]:
financial['Category'].value_counts()

Category
Health & Fitness    1417
Rent                1373
Utilities           1327
Shopping            1249
Entertainment       1230
Food & Drink        1200
Travel              1174
Name: count, dtype: int64

In [14]:
financial['employment_status'].value_counts()

employment_status
Employed         5373
Self-employed    1835
Student           896
Unemployed        866
Name: count, dtype: int64

In [15]:
ibm_sample.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
User,5000.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Card,5000.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Year,5000.0,NaN,NaN,NaN,2009.7648,5.142448,2002.0,2005.0,2010.0,2014.0,2020.0
Month,5000.0,NaN,NaN,NaN,6.7098,3.484596,1.0,4.0,7.0,10.0,12.0
Day,5000.0,NaN,NaN,NaN,15.8292,8.792445,1.0,8.0,16.0,23.0,31.0
Time,5000,789,06:16,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Amount,5000,4026,$140.00,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Use Chip,5000,3,Swipe Transaction,3732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Merchant Name,5000.0,NaN,NaN,NaN,667554873247362432.0,4041932990435706368.0,-9092677072201095168.0,-1288082279022882048.0,-34551508091458520.0,4060646732831064576.0,9137769173591184384.0
Merchant City,5000,161,La Verne,2742,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
ibm_sample["Use Chip"].value_counts()

Use Chip
Swipe Transaction     3732
Chip Transaction       975
Online Transaction     293
Name: count, dtype: int64

In [17]:
ibm_sample["MCC"].nunique()

71

In [18]:
paysim_sample.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
step,5000.0,NaN,NaN,NaN,1.8914,1.175705,1.0,1.0,1.0,3.0,5.0
type,5000,5,PAYMENT,2587,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,5000.0,NaN,NaN,NaN,101517.56378,300962.329334,6.42,3734.9225,10958.55,108604.585,10000000.0
nameOrig,5000,5000,C1231006815,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceOrg,5000.0,NaN,NaN,NaN,1042543.400612,2301985.714553,0.0,572.25,26550.555,295718.28,12930418.44
newbalanceOrig,5000.0,NaN,NaN,NaN,1064337.794818,2343680.002691,0.0,0.0,18141.855,300320.69,13010502.78
nameDest,5000,3006,C985934102,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceDest,5000.0,NaN,NaN,NaN,918153.634206,2541758.977139,0.0,0.0,0.0,363847.48,19516116.38
newbalanceDest,5000.0,NaN,NaN,NaN,1064912.46547,2921891.971154,0.0,0.0,0.0,270110.78,19169204.93
isFraud,5000.0,NaN,NaN,NaN,0.008,0.089093,0.0,0.0,0.0,0.0,1.0


In [19]:
paysim_sample["type"].value_counts()

type
PAYMENT     2587
CASH_IN     1082
CASH_OUT     631
TRANSFER     437
DEBIT        263
Name: count, dtype: int64

In [20]:
print(ibm_sample["Amount"].dtype)
print(ibm_sample["Amount"].head(10))
print(ibm_sample["Amount"].map(type).value_counts())

str
0    $134.09
1     $38.48
2    $120.34
3    $128.95
4    $104.71
5     $86.19
6     $93.84
7    $123.50
8     $61.72
9     $57.10
Name: Amount, dtype: str
Amount
<class 'str'>    5000
Name: count, dtype: int64


In [21]:
ibm_sample_clean = ibm_sample.copy()

ibm_sample_clean["Amount"] = (
    ibm_sample_clean["Amount"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

ibm_sample_clean["Amount"] = pd.to_numeric(
    ibm_sample_clean["Amount"],
    errors="coerce"
)

print(ibm_sample_clean["Amount"].dtype)
print(ibm_sample_clean["Amount"].head())

float64
0    134.09
1     38.48
2    120.34
3    128.95
4    104.71
Name: Amount, dtype: float64


In [22]:
from src.data.aggregation import aggregate_ibm_chunk
ibm_features_test = aggregate_ibm_chunk(ibm_sample_clean)

display(ibm_features_test.head())

,User,total_spending,transaction_count
0,0,331216.3,5000


In [23]:
import importlib

import src.data.data_cleaning as cleaning
importlib.reload(cleaning)

<module 'src.data.data_cleaning' from '/Users/ms/Desktop/FinSense/src/data/data_cleaning.py'>

In [24]:
clean_ibm_chunk = cleaning.clean_ibm_chunk
ibm_clean_test = clean_ibm_chunk(ibm_sample)
print(ibm_clean_test["Amount"].dtype)
print(ibm_clean_test["Amount"].head())

float64
0    134.09
1     38.48
2    120.34
3    128.95
4    104.71
Name: Amount, dtype: float64


In [25]:
import src.data.aggregation as aggregation
importlib.reload(aggregation)

aggregate_ibm_chunk = aggregation.aggregate_ibm_chunk

ibm_features_test = aggregate_ibm_chunk(ibm_clean_test)

print(ibm_features_test.shape)
display(ibm_features_test.head())

(1, 3)


,User,total_spending,transaction_count
0,0,331216.3,5000


In [26]:
import src.data.aggregation as aggregation
combine_ibm_aggregates = aggregation.combine_ibm_aggregates
chunk1_result = pd.DataFrame({
    "User": ["A", "B"],
    "total_spending": [100, 200],
    "transaction_count": [2, 1]
})

chunk2_result = pd.DataFrame({
    "User": ["A", "C"],
    "total_spending": [300, 150],
    "transaction_count": [3, 1]
})

final_test = combine_ibm_aggregates(
    [chunk1_result, chunk2_result]
)

display(final_test)

,User,total_spending,transaction_count,average_transaction
0,A,400,5,80.0
1,B,200,1,200.0
2,C,150,1,150.0


In [27]:
import pandas as pd

from src.data.data_cleaning import clean_ibm_chunk
from src.data.aggregation import (
    aggregate_ibm_chunk,
    combine_ibm_aggregates
)
from src.data.chunk_processing import process_chunks

In [28]:
IBM_PATH = "/Users/ms/Desktop/FinSense/data/raw/IBM/credit_card_transactions-ibm_v2.csv"

reader = pd.read_csv(
    IBM_PATH,
    chunksize=10_000
)

In [29]:
partial_results = process_chunks(
    reader,
    lambda chunk: aggregate_ibm_chunk(
        clean_ibm_chunk(chunk)
    )
)

Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Processing chunk 25...
Processing chunk 26...
Processing chunk 27...
Processing chunk 28...
Processing chunk 29...
Processing chunk 30...
Processing chunk 31...
Processing chunk 32...
Processing chunk 33...
Processing chunk 34...
Processing chunk 35...
Processing chunk 36...
Processing chunk 37...
Processing chunk 38...
Processing chunk 39...
Processing chunk 40...
Processing chunk 41...
Processing chunk 42...
Processing chunk 43...
Processing chunk 44.

In [82]:
final_test = combine_ibm_aggregates(partial_results)

display(final_test.head())


,User,total_spending,transaction_count,average_transaction
0,0,1622991.69,19963,81.299989
1,1,723491.89,8919,81.118050
2,2,1475933.36,41978,35.159687
3,3,1186497.51,10117,117.277603
4,4,1798790.91,18542,97.011698


(2000, 4)

In [31]:
import importlib
import src.data.feature_engineering as feature_engineering

importlib.reload(feature_engineering)

<module 'src.data.feature_engineering' from '/Users/ms/Desktop/FinSense/src/data/feature_engineering.py'>

In [32]:
from src.data.feature_engineering import engineer_ibm_features
ibm_features = engineer_ibm_features(ibm_clean_test)

print(ibm_features.shape)
display(ibm_features.head())

(1, 16)


,User,total_spending,transaction_count,average_transaction,max_transaction,unique_merchants,unique_cities,unique_mcc,active_days,monthly_spending,night_transaction_ratio,weekend_transaction_ratio,online_transaction_ratio,swipe_transaction_ratio,chip_transaction_ratio,largest_category_share
0,0,331216.3,5000,66.24326,1049.82,286,161,71,3296,27601.358333,0.0486,0.2878,0.0586,0.7464,0.195,0.196031


In [33]:
ibm_features = engineer_ibm_features(ibm_clean_test)

print(ibm_features.shape)
display(ibm_features.head())
print(ibm_features.dtypes)

(1, 16)


,User,total_spending,transaction_count,average_transaction,max_transaction,unique_merchants,unique_cities,unique_mcc,active_days,monthly_spending,night_transaction_ratio,weekend_transaction_ratio,online_transaction_ratio,swipe_transaction_ratio,chip_transaction_ratio,largest_category_share
0,0,331216.3,5000,66.24326,1049.82,286,161,71,3296,27601.358333,0.0486,0.2878,0.0586,0.7464,0.195,0.196031


User                           int64
total_spending               float64
transaction_count              int64
average_transaction          float64
max_transaction              float64
unique_merchants               int64
unique_cities                  int64
unique_mcc                     int64
active_days                    int64
monthly_spending             float64
night_transaction_ratio      float64
weekend_transaction_ratio    float64
online_transaction_ratio     float64
swipe_transaction_ratio      float64
chip_transaction_ratio       float64
largest_category_share       float64
dtype: object


In [34]:
print(ibm_clean_test["Use Chip"].value_counts())


Use Chip
Swipe Transaction     3732
Chip Transaction       975
Online Transaction     293
Name: count, dtype: int64


In [35]:
print(ibm_features.shape)
display(ibm_features.head())
print(ibm_features.isna().sum())

(1, 16)


,User,total_spending,transaction_count,average_transaction,max_transaction,unique_merchants,unique_cities,unique_mcc,active_days,monthly_spending,night_transaction_ratio,weekend_transaction_ratio,online_transaction_ratio,swipe_transaction_ratio,chip_transaction_ratio,largest_category_share
0,0,331216.3,5000,66.24326,1049.82,286,161,71,3296,27601.358333,0.0486,0.2878,0.0586,0.7464,0.195,0.196031


User                         0
total_spending               0
transaction_count            0
average_transaction          0
max_transaction              0
unique_merchants             0
unique_cities                0
unique_mcc                   0
active_days                  0
monthly_spending             0
night_transaction_ratio      0
weekend_transaction_ratio    0
online_transaction_ratio     0
swipe_transaction_ratio      0
chip_transaction_ratio       0
largest_category_share       0
dtype: int64


In [36]:
os.makedirs("data/interim", exist_ok=True)

In [37]:
ibm_features.to_csv(
    "data/interim/ibm_behavioral_features.csv",
    index=False
)

In [38]:
print(ibm_features.shape)
print(ibm_features.columns.tolist())

(1, 16)
['User', 'total_spending', 'transaction_count', 'average_transaction', 'max_transaction', 'unique_merchants', 'unique_cities', 'unique_mcc', 'active_days', 'monthly_spending', 'night_transaction_ratio', 'weekend_transaction_ratio', 'online_transaction_ratio', 'swipe_transaction_ratio', 'chip_transaction_ratio', 'largest_category_share']


Phase 2B : Financial Dataset


In [39]:
financial = pd.read_csv('/Users/ms/Desktop/FinSense/data/raw/financial/cleaned_financial_data.csv')
financial.head()

,user_id,Category,Amount,age,gender,education_level,employment_status,job_title,monthly_income_usd,monthly_expenses_usd,...,loan_type,loan_amount_usd,loan_term_months,monthly_emi_usd,loan_interest_rate_pct,debt_to_income_ratio,credit_score,savings_to_income_ratio,region,record_date
0,U32234,Health & Fitness,282.25,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
1,U32234,Food & Drink,1082.58,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
2,U32234,Entertainment,1031.44,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
3,U32234,Shopping,1782.34,50,Female,Bachelor,Employed,Salesperson,4164.52,3552.28,...,No Loan,0.00,0,0.00,0.00,0.00,366,2.51,Europe,2023-03-31
4,U28077,Entertainment,463.71,43,Female,Bachelor,Employed,Driver,3231.62,1715.77,...,Business,493968.77,24,22787.31,9.97,7.05,698,4.91,North America,2021-12-23


In [40]:
financial.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['user_id', 'Category', 'Amount', 'age', 'gender', 'education_level',
       'employment_status', 'job_title', 'monthly_income_usd',
       'monthly_expenses_usd', 'savings_usd', 'has_loan', 'loan_type',
       'loan_amount_usd', 'loan_term_months', 'monthly_emi_usd',
       'loan_interest_rate_pct', 'debt_to_income_ratio', 'credit_score',
       'savings_to_income_ratio', 'region', 'record_date'],
      dtype='str')>

In [41]:
print((financial["monthly_income_usd"] == 0).sum())

0


In [42]:
print(
    financial[
        [
            "monthly_income_usd",
            "monthly_expenses_usd",
            "savings_usd",
            "monthly_emi_usd",
            "debt_to_income_ratio",
            "credit_score"
        ]
    ].describe()
)

       monthly_income_usd  monthly_expenses_usd   savings_usd  \
count         8970.000000           8970.000000  8.970000e+03   
mean          4825.722931           3073.900326  2.869629e+05   
std           1823.350735           1433.385067  2.116524e+05   
min            500.000000            150.410000  7.248600e+02   
25%           3547.430000           1987.760000  1.188064e+05   
50%           4766.050000           2958.350000  2.494065e+05   
75%           6070.690000           3956.770000  4.059135e+05   
max          10222.850000           8833.990000  1.154563e+06   

       monthly_emi_usd  debt_to_income_ratio  credit_score  
count      8970.000000           8970.000000   8970.000000  
mean       3056.318303              0.792845    571.919844  
std        6575.219791              2.331759    160.314954  
min           0.000000              0.000000    300.000000  
25%           0.000000              0.000000    429.000000  
50%           0.000000              0.000000    

In [43]:
financial["disposable_income_usd"] = (
    financial["monthly_income_usd"]
    - financial["monthly_expenses_usd"]
)

In [44]:
financial["expense_to_income_ratio"] = (
    financial["monthly_expenses_usd"]
    / financial["monthly_income_usd"]
)

In [45]:
financial["emi_to_income_ratio"] = (
    financial["monthly_emi_usd"]
    / financial["monthly_income_usd"]
)

In [46]:
display(
    financial[
        [
            "monthly_income_usd",
            "monthly_expenses_usd",
            "monthly_emi_usd",
            "disposable_income_usd",
            "expense_to_income_ratio",
            "emi_to_income_ratio"
        ]
    ].head()
)

,monthly_income_usd,monthly_expenses_usd,monthly_emi_usd,disposable_income_usd,expense_to_income_ratio,emi_to_income_ratio
0,4164.52,3552.28,0.00,612.24,0.852987,0.000000
1,4164.52,3552.28,0.00,612.24,0.852987,0.000000
2,4164.52,3552.28,0.00,612.24,0.852987,0.000000
3,4164.52,3552.28,0.00,612.24,0.852987,0.000000
4,3231.62,1715.77,22787.31,1515.85,0.530932,7.051358


In [47]:
financial["savings_months"] = (
    financial["savings_usd"]
    / financial["monthly_expenses_usd"]
)

In [48]:
financial["has_active_loan"] = (
    financial["has_loan"] == "Yes"
)

In [49]:
financial["loan_to_income_ratio"] = (
    financial["loan_amount_usd"]
    / financial["monthly_income_usd"]
)

In [50]:
display(
    financial[
        [
            "has_loan",
            "loan_amount_usd",
            "monthly_income_usd",
            "loan_to_income_ratio"
        ]
    ].head()
)

,has_loan,loan_amount_usd,monthly_income_usd,loan_to_income_ratio
0,No,0.00,4164.52,0.000000
1,No,0.00,4164.52,0.000000
2,No,0.00,4164.52,0.000000
3,No,0.00,4164.52,0.000000
4,Yes,493968.77,3231.62,152.854844


In [51]:
financial_profile = financial[
    [
        "user_id",
        "age",
        "monthly_income_usd",
        "monthly_expenses_usd",
        "savings_usd",
        "has_active_loan",
        "loan_amount_usd",
        "monthly_emi_usd",
        "loan_interest_rate_pct",
        "credit_score",
        "debt_to_income_ratio",
        "savings_to_income_ratio",

        # engineered
        "disposable_income_usd",
        "expense_to_income_ratio",
        "emi_to_income_ratio",
        "savings_months",
        "loan_to_income_ratio",
    ]
].copy()

In [52]:
print(financial_profile.shape)
display(financial_profile.head())
print(financial_profile.isna().sum().sum())

(8970, 17)


,user_id,age,monthly_income_usd,monthly_expenses_usd,savings_usd,has_active_loan,loan_amount_usd,monthly_emi_usd,loan_interest_rate_pct,credit_score,debt_to_income_ratio,savings_to_income_ratio,disposable_income_usd,expense_to_income_ratio,emi_to_income_ratio,savings_months,loan_to_income_ratio
0,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
1,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
2,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
3,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
4,U28077,43,3231.62,1715.77,190553.69,True,493968.77,22787.31,9.97,698,7.05,4.91,1515.85,0.530932,7.051358,111.060160,152.854844


0


In [53]:
financial_profile.to_csv(
    "data/interim/financial_profile_features.csv",
    index=False
)

In [54]:
print(paysim_sample.columns.tolist())
display(paysim_sample.head())
print(paysim_sample.dtypes)

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


In [55]:
print(paysim_sample['type'].value_counts())

type
PAYMENT     2587
CASH_IN     1082
CASH_OUT     631
TRANSFER     437
DEBIT        263
Name: count, dtype: int64


In [56]:
transaction_count = paysim_sample.groupby('nameOrig')['amount'].count()
print(transaction_count)

nameOrig
C10001825      1
C1000600589    1
C1002041276    1
C1002872041    1
C1002911155    1
              ..
C998242313     1
C998574047     1
C999427036     1
C999647352     1
C999864329     1
Name: amount, Length: 5000, dtype: int64


In [57]:
total_transaction_amount = paysim_sample.groupby('nameOrig')['amount'].sum()
print(total_transaction_amount)


nameOrig
C10001825        2655.04
C1000600589     67017.13
C1002041276      2444.49
C1002872041      6064.70
C1002911155      1246.74
                 ...    
C998242313     182861.45
C998574047       2273.88
C999427036        772.01
C999647352       4531.55
C999864329       2181.96
Name: amount, Length: 5000, dtype: float64


In [58]:
outflow = paysim_sample[
    paysim_sample['type'].isin(
        ['PAYMENT','CASH_OUT','TRANSFER','DEBIT']
    )
]

total_outflow = outflow.groupby('nameOrig')['amount'].sum()
print(total_outflow)

nameOrig
C10001825      2655.04
C1002041276    2444.49
C1002872041    6064.70
C1002911155    1246.74
C1003206025     871.75
                ...   
C998015174     9059.91
C998574047     2273.88
C999427036      772.01
C999647352     4531.55
C999864329     2181.96
Name: amount, Length: 3918, dtype: float64


In [59]:
inflow = paysim_sample[
    paysim_sample["type"].str.strip() == "CASH_IN"
]

total_inflow = (
    inflow
    .groupby("nameOrig")["amount"]
    .sum()
)

print(total_inflow.head())
print(len(total_inflow))

nameOrig
C1000600589     67017.13
C1006297128    344106.28
C100708535     377287.80
C1011599868    198443.49
C1015659282    188071.25
Name: amount, dtype: float64
1082


In [60]:
import importlib
import src.data.feature_engineering as fe

importlib.reload(fe)

<module 'src.data.feature_engineering' from '/Users/ms/Desktop/FinSense/src/data/feature_engineering.py'>

In [61]:
paysim_features = fe.engineer_paysim_features(paysim_sample)

print(paysim_features.shape)
display(paysim_features.head())

(5000, 25)


,nameOrig,transaction_count,total_transaction_amount,average_transaction,max_transaction,transaction_volatility,total_outflow,total_inflow,net_cash_flow,payment_ratio,...,average_balance_change,balance_depletion_ratio,active_steps,transaction_frequency,max_transactions_in_step,average_balance_utilization,high_balance_utilization_ratio,average_balance_error,max_balance_error,balance_mismatch_ratio
0,C10001825,1,2655.04,2655.04,2655.04,0.0,2655.04,0.00,0.0,0.0,...,2655.04,0.0,1,1.0,1,0.125155,0.0,0.000000e+00,0.000000e+00,0.0
1,C1000600589,1,67017.13,67017.13,67017.13,0.0,0.00,67017.13,0.0,0.0,...,67017.13,0.0,1,1.0,1,0.016998,0.0,0.000000e+00,0.000000e+00,0.0
2,C1002041276,1,2444.49,2444.49,2444.49,0.0,2444.49,0.00,0.0,1.0,...,2444.49,0.0,1,1.0,1,0.077716,0.0,3.637979e-12,3.637979e-12,1.0
3,C1002872041,1,6064.70,6064.70,6064.70,0.0,6064.70,0.00,0.0,1.0,...,6064.70,0.0,1,1.0,1,0.194606,0.0,0.000000e+00,0.000000e+00,0.0
4,C1002911155,1,1246.74,1246.74,1246.74,0.0,1246.74,0.00,0.0,1.0,...,1246.74,0.0,1,1.0,1,0.849278,1.0,0.000000e+00,0.000000e+00,0.0


In [62]:
paysim_features = paysim_features.fillna(0)

paysim_features = paysim_features.replace(
    [float("inf"), float("-inf")],
    0
)

In [63]:
print(paysim_features.isna().sum())
print(paysim_features.isin([float("inf"), float("-inf")]).sum())


nameOrig                          0
transaction_count                 0
total_transaction_amount          0
average_transaction               0
max_transaction                   0
transaction_volatility            0
total_outflow                     0
total_inflow                      0
net_cash_flow                     0
payment_ratio                     0
transfer_ratio                    0
cash_out_ratio                    0
debit_ratio                       0
cash_in_ratio                     0
large_transaction_ratio           0
average_balance_change            0
balance_depletion_ratio           0
active_steps                      0
transaction_frequency             0
max_transactions_in_step          0
average_balance_utilization       0
high_balance_utilization_ratio    0
average_balance_error             0
max_balance_error                 0
balance_mismatch_ratio            0
dtype: int64
nameOrig                          0
transaction_count                 0
total_transacti

In [64]:
print(paysim_features.shape)
print(paysim_features.isna().sum().sum())
print(paysim_features.isin([float("inf"), float("-inf")]).sum().sum())


(5000, 25)
0
0


In [65]:
import importlib
import src.data.feature_engineering as fe

importlib.reload(fe)

paysim_target = fe.create_paysim_target(paysim_sample)

display(paysim_target.head())

,nameOrig,fraud_count,fraud_rate,is_fraud
0,C10001825,0,0.0,0
1,C1000600589,0,0.0,0
2,C1002041276,0,0.0,0
3,C1002872041,0,0.0,0
4,C1002911155,0,0.0,0


In [66]:
paysim_model_data = paysim_features.merge(
    paysim_target,
    on="nameOrig",
    how="left"
)

print(paysim_model_data.shape)
display(paysim_model_data.head())

(5000, 28)


,nameOrig,transaction_count,total_transaction_amount,average_transaction,max_transaction,transaction_volatility,total_outflow,total_inflow,net_cash_flow,payment_ratio,...,transaction_frequency,max_transactions_in_step,average_balance_utilization,high_balance_utilization_ratio,average_balance_error,max_balance_error,balance_mismatch_ratio,fraud_count,fraud_rate,is_fraud
0,C10001825,1,2655.04,2655.04,2655.04,0.0,2655.04,0.00,0.0,0.0,...,1.0,1,0.125155,0.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0
1,C1000600589,1,67017.13,67017.13,67017.13,0.0,0.00,67017.13,0.0,0.0,...,1.0,1,0.016998,0.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0
2,C1002041276,1,2444.49,2444.49,2444.49,0.0,2444.49,0.00,0.0,1.0,...,1.0,1,0.077716,0.0,3.637979e-12,3.637979e-12,1.0,0,0.0,0
3,C1002872041,1,6064.70,6064.70,6064.70,0.0,6064.70,0.00,0.0,1.0,...,1.0,1,0.194606,0.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0
4,C1002911155,1,1246.74,1246.74,1246.74,0.0,1246.74,0.00,0.0,1.0,...,1.0,1,0.849278,1.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0


In [67]:
print(paysim_model_data["is_fraud"].value_counts())

is_fraud
0    4960
1      40
Name: count, dtype: int64


In [68]:
print(paysim_model_data.shape)
display(paysim_model_data.head())

(5000, 28)


,nameOrig,transaction_count,total_transaction_amount,average_transaction,max_transaction,transaction_volatility,total_outflow,total_inflow,net_cash_flow,payment_ratio,...,transaction_frequency,max_transactions_in_step,average_balance_utilization,high_balance_utilization_ratio,average_balance_error,max_balance_error,balance_mismatch_ratio,fraud_count,fraud_rate,is_fraud
0,C10001825,1,2655.04,2655.04,2655.04,0.0,2655.04,0.00,0.0,0.0,...,1.0,1,0.125155,0.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0
1,C1000600589,1,67017.13,67017.13,67017.13,0.0,0.00,67017.13,0.0,0.0,...,1.0,1,0.016998,0.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0
2,C1002041276,1,2444.49,2444.49,2444.49,0.0,2444.49,0.00,0.0,1.0,...,1.0,1,0.077716,0.0,3.637979e-12,3.637979e-12,1.0,0,0.0,0
3,C1002872041,1,6064.70,6064.70,6064.70,0.0,6064.70,0.00,0.0,1.0,...,1.0,1,0.194606,0.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0
4,C1002911155,1,1246.74,1246.74,1246.74,0.0,1246.74,0.00,0.0,1.0,...,1.0,1,0.849278,1.0,0.000000e+00,0.000000e+00,0.0,0,0.0,0


In [69]:
X = paysim_model_data.drop(
    columns=["nameOrig", "fraud_count", "fraud_rate", "is_fraud"]
)

y = paysim_model_data["is_fraud"]

In [70]:
print(X.columns.tolist())

['transaction_count', 'total_transaction_amount', 'average_transaction', 'max_transaction', 'transaction_volatility', 'total_outflow', 'total_inflow', 'net_cash_flow', 'payment_ratio', 'transfer_ratio', 'cash_out_ratio', 'debit_ratio', 'cash_in_ratio', 'large_transaction_ratio', 'average_balance_change', 'balance_depletion_ratio', 'active_steps', 'transaction_frequency', 'max_transactions_in_step', 'average_balance_utilization', 'high_balance_utilization_ratio', 'average_balance_error', 'max_balance_error', 'balance_mismatch_ratio']


In [71]:
X.nunique().sort_values()

transaction_count                    1
max_transactions_in_step             1
transaction_frequency                1
transaction_volatility               1
active_steps                         1
net_cash_flow                        1
large_transaction_ratio              1
high_balance_utilization_ratio       2
balance_depletion_ratio              2
cash_in_ratio                        2
debit_ratio                          2
transfer_ratio                       2
payment_ratio                        2
cash_out_ratio                       2
balance_mismatch_ratio               2
total_inflow                      1083
max_balance_error                 1830
average_balance_error             1830
average_balance_change            3862
total_outflow                     3898
average_balance_utilization       3898
max_transaction                   4979
average_transaction               4979
total_transaction_amount          4979
dtype: int64


std

If:

std = 0

the feature is constant → useless.

If:

std > 0

there is variation → potentially useful.

In [72]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_count,5000.0,1.000000,0.000000,1.00,1.0000,1.000000e+00,1.0000,1.00
total_transaction_amount,5000.0,101517.563780,300962.329334,6.42,3734.9225,1.095855e+04,108604.5850,10000000.00
average_transaction,5000.0,101517.563780,300962.329334,6.42,3734.9225,1.095855e+04,108604.5850,10000000.00
max_transaction,5000.0,101517.563780,300962.329334,6.42,3734.9225,1.095855e+04,108604.5850,10000000.00
transaction_volatility,5000.0,0.000000,0.000000,0.00,0.0000,0.000000e+00,0.0000,0.00
total_outflow,5000.0,65338.941164,293777.393550,0.00,502.2350,4.382005e+03,14199.9900,10000000.00
total_inflow,5000.0,36178.622616,94878.123106,0.00,0.0000,0.000000e+00,0.0000,1030100.15
net_cash_flow,5000.0,0.000000,0.000000,0.00,0.0000,0.000000e+00,0.0000,0.00
payment_ratio,5000.0,0.517400,0.499747,0.00,0.0000,1.000000e+00,1.0000,1.00
transfer_ratio,5000.0,0.087400,0.282449,0.00,0.0000,0.000000e+00,0.0000,1.00


In [73]:
correlations = (
    paysim_model_data
    .corr(numeric_only=True)["is_fraud"]
    .sort_values()
)

print(correlations)

payment_ratio                    -0.092984
balance_mismatch_ratio           -0.084876
cash_in_ratio                    -0.047192
total_inflow                     -0.034247
debit_ratio                      -0.021160
max_balance_error                -0.017403
average_balance_error            -0.017403
cash_out_ratio                    0.107847
balance_depletion_ratio           0.115938
transfer_ratio                    0.123247
high_balance_utilization_ratio    0.167236
total_transaction_amount          0.228089
max_transaction                   0.228089
average_transaction               0.228089
total_outflow                     0.244728
average_balance_change            0.309065
fraud_rate                        1.000000
fraud_count                       1.000000
is_fraud                          1.000000
transaction_count                      NaN
transaction_volatility                 NaN
net_cash_flow                          NaN
large_transaction_ratio                NaN
active_step

In [74]:
print(y.value_counts())
print(y.value_counts(normalize=True))

is_fraud
0    4960
1      40
Name: count, dtype: int64
is_fraud
0    0.992
1    0.008
Name: proportion, dtype: float64


That's class imbalance.

And that means later we should not judge the model using accuracy alone.

In [85]:
import pandas as pd

# 1. Load the two datasets you ALREADY saved earlier in this notebook
financial_profile = pd.read_csv("data/interim/financial_profile_features.csv")
ibm_features      = pd.read_csv("data/interim/ibm_behavioral_features.csv")

# 2. Since paysim_features is still in your notebook memory, save it now!
paysim_features.to_csv("data/interim/paysim_features.csv", index=False)

# Now you can load it back anytime:
paysim_features = pd.read_csv("data/interim/paysim_features.csv")

print("Successfully loaded all 3 final feature datasets!")


Successfully loaded all 3 final feature datasets!


In [86]:
print("IBM")
print(ibm_features.shape)
print(ibm_features.columns.tolist())

IBM
(1, 16)
['User', 'total_spending', 'transaction_count', 'average_transaction', 'max_transaction', 'unique_merchants', 'unique_cities', 'unique_mcc', 'active_days', 'monthly_spending', 'night_transaction_ratio', 'weekend_transaction_ratio', 'online_transaction_ratio', 'swipe_transaction_ratio', 'chip_transaction_ratio', 'largest_category_share']


In [87]:
print("\nPaySim")
print(paysim_features.shape)
print(paysim_features.columns.tolist())
print(paysim_features["nameOrig"].nunique())


PaySim
(5000, 25)
['nameOrig', 'transaction_count', 'total_transaction_amount', 'average_transaction', 'max_transaction', 'transaction_volatility', 'total_outflow', 'total_inflow', 'net_cash_flow', 'payment_ratio', 'transfer_ratio', 'cash_out_ratio', 'debit_ratio', 'cash_in_ratio', 'large_transaction_ratio', 'average_balance_change', 'balance_depletion_ratio', 'active_steps', 'transaction_frequency', 'max_transactions_in_step', 'average_balance_utilization', 'high_balance_utilization_ratio', 'average_balance_error', 'max_balance_error', 'balance_mismatch_ratio']
5000


In [88]:
print("\nFinancial Profile")
print(financial_profile.shape)
print(financial_profile.columns.tolist())
print(financial_profile["user_id"].nunique())


Financial Profile
(8970, 17)
['user_id', 'age', 'monthly_income_usd', 'monthly_expenses_usd', 'savings_usd', 'has_active_loan', 'loan_amount_usd', 'monthly_emi_usd', 'loan_interest_rate_pct', 'credit_score', 'debt_to_income_ratio', 'savings_to_income_ratio', 'disposable_income_usd', 'expense_to_income_ratio', 'emi_to_income_ratio', 'savings_months', 'loan_to_income_ratio']
2746


In [91]:
import pandas as pd
import src.data.feature_engineering as fe
import src.data.data_cleaning as dc  # Added this import

# 1. Load the FULL datasets
print("Loading full datasets...")
full_ibm = pd.read_parquet("../data/processed/ibm_transactions.parquet")
full_paysim = pd.read_parquet("../data/processed/paysim.parquet")

# 2. **CRITICAL FIX**: Clean the IBM data to convert '$134.09' strings into 134.09 numbers!
print("Cleaning IBM data...")
full_ibm = dc.clean_ibm_chunk(full_ibm)

# 3. Apply your functions to the FULL datasets
# (This processes 30M total rows, so it might take 1-2 minutes)
print("Engineering full IBM features (24M rows)...")
full_ibm_features = fe.engineer_ibm_features(full_ibm)

print("Engineering full PaySim features (6M rows)...")
full_paysim_features = fe.engineer_paysim_features(full_paysim)

# 4. Save the true FULL engineered feature matrices for deep learning!
full_ibm_features.to_csv("../data/processed/full_ibm_features.csv", index=False)
full_paysim_features.to_csv("../data/processed/full_paysim_features.csv", index=False)

print("Done! Real shapes:")
print("IBM:", full_ibm_features.shape)
print("PaySim:", full_paysim_features.shape)


Loading full datasets...
Cleaning IBM data...
Engineering full IBM features (24M rows)...
Engineering full PaySim features (6M rows)...
Done! Real shapes:
IBM: (2000, 16)
PaySim: (6353307, 25)


In [92]:
full_ibm_features.head()

,User,total_spending,transaction_count,average_transaction,max_transaction,unique_merchants,unique_cities,unique_mcc,active_days,monthly_spending,night_transaction_ratio,weekend_transaction_ratio,online_transaction_ratio,swipe_transaction_ratio,chip_transaction_ratio,largest_category_share
0,0,1622991.69,19963,81.299989,1409.40,552,295,94,6227,135249.307500,0.053599,0.288734,0.065872,0.793468,0.140660,0.168463
1,1,723491.89,8919,81.118050,3750.60,337,263,90,4517,60290.990833,0.119296,0.283552,0.074224,0.638076,0.287700,0.109446
2,2,1475933.36,41978,35.159687,1195.57,727,314,91,6558,122994.446667,0.261994,0.285840,0.390752,0.483968,0.125280,0.465422
3,3,1186497.51,10117,117.277603,6820.20,655,392,96,4024,98874.792500,0.092221,0.282692,0.096372,0.510922,0.392705,0.120337
4,4,1798790.91,18542,97.011698,3613.22,624,402,94,6799,149899.242500,0.036889,0.281793,0.120268,0.673876,0.205857,0.099871


In [93]:
full_paysim_features.head()

,nameOrig,transaction_count,total_transaction_amount,average_transaction,max_transaction,transaction_volatility,total_outflow,total_inflow,net_cash_flow,payment_ratio,...,average_balance_change,balance_depletion_ratio,active_steps,transaction_frequency,max_transactions_in_step,average_balance_utilization,high_balance_utilization_ratio,average_balance_error,max_balance_error,balance_mismatch_ratio
0,C1000000639,1,244486.46,244486.46,244486.46,0.0,244486.46,0.00,0.0,0.0,...,8946.00,1.0,1,1.0,1,27.329137,1.0,235540.46,235540.46,1.0
1,C1000001337,1,3170.28,3170.28,3170.28,0.0,3170.28,0.00,0.0,1.0,...,3170.28,0.0,1,1.0,1,0.054576,0.0,0.00,0.00,0.0
2,C1000001725,1,8424.74,8424.74,8424.74,0.0,8424.74,0.00,0.0,1.0,...,783.00,1.0,1,1.0,1,10.759566,1.0,7641.74,7641.74,1.0
3,C1000002591,1,261877.19,261877.19,261877.19,0.0,0.00,261877.19,0.0,0.0,...,261877.19,0.0,1,1.0,1,34.47567,1.0,0.00,0.00,0.0
4,C1000003372,1,20528.65,20528.65,20528.65,0.0,0.00,20528.65,0.0,0.0,...,20528.65,0.0,1,1.0,1,0.008917,0.0,0.00,0.00,0.0


In [94]:
financial_profile.head()

,user_id,age,monthly_income_usd,monthly_expenses_usd,savings_usd,has_active_loan,loan_amount_usd,monthly_emi_usd,loan_interest_rate_pct,credit_score,debt_to_income_ratio,savings_to_income_ratio,disposable_income_usd,expense_to_income_ratio,emi_to_income_ratio,savings_months,loan_to_income_ratio
0,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
1,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
2,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
3,U32234,50,4164.52,3552.28,125636.23,False,0.00,0.00,0.00,366,0.00,2.51,612.24,0.852987,0.000000,35.367772,0.000000
4,U28077,43,3231.62,1715.77,190553.69,True,493968.77,22787.31,9.97,698,7.05,4.91,1515.85,0.530932,7.051358,111.060160,152.854844
